#### Baseline Forecast

One of the first things we need to establish is the baseline.

Baseline can be as sophisticated as we want it to be. We look at a few classical techniques that can be used as baselines. Some of the techniques are mature, can be applied with little effort using open-source libraries to implement them. There can be many types of problems and datasets where it is difficult to beat the baseline techniques.

We setup test harness, generate strong baseline forecasts, and assess the forecastability of a time series.

Test harness, the collection of code and inputs considered to test the program under various situations. In ML is a set of code and data used to evaluate algorithms. It is important to set-up test harness to eval all future algorithms in a standard and quick way. Use holdout (test) and validation datasets. Choose an evaluation metric, Mean Absolute Error and Mean Squared Error. In time series realm there are scores of metrics with no real consensus on which ones to use.

We consider these metrics to measure the forecast.
- Mean Absolute Error (MAE), the average of the unsigned error between the forecast at timestep $t(f_t)$ and the observed value at time $t(y_t)$.
- Mean Squared Error (MSE), the average of the squared error between the forecast $(f_t)$
and observed $(y_t)$ values.
- Mean Absolute Scaled Error (MASE) is slightly more complicated than MSE and MAE but gives us a slightly better measure to overcome the scale-dependent nature of the previous two measures.

If we have multiple time series with different average values, MAE and MSE will show higher errors for the high-value time series as opposed to the low-valued time series. MASE overcomes this by scaling the errors based on the in-sample MAE from the naïve forecasting method. Intuitively, MASE gives us the measure of how much better our forecast is as compared to the naïve forecast.

- Forecast Bias (FB) metric with slightly different aspects from the other metrics help assess the correctness of the forecast, irrespective of the direction of the error, forecast bias lets us understand the overall bias in the model. Forecast bias is a metric that helps us understand whether the forecast is continuously over-forecasting or under-forecasting.

In [ ]:
import pandas as pd
import numpy as np
import os, pathlib
from pathlib import Path

In [ ]:
block_df = pd.read_parquet('hhblock_df.parquet')

In [ ]:
exp_block_df = compact_to_expanded(
    block_df, timeseries_col='energy_consumption',
    static_cols=['frequency', 'series_length', 'stdorToU', 'Acorn', 'Acorn_grouped'],
    time_varying_cols=['pressure', 'apparentTemperature', 'windSpeed', 'precipType', 'icon', 'humidity', 'summary'],
    ts_identifier='LCLid')

  0%|          | 0/799 [00:00<?, ?it/s]

In [ ]:
exp_block_df.head(1)

,timestamp,LCLid,energy_consumption,frequency,series_length,stdorToU,Acorn,Acorn_grouped,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,2012-10-13,MAC000002,0.263,30min,24144,Std,ACORN-A,Affluent,1007.7,7.55,2.28,rain,clear-night,0.84,Clear


Train, Test and Validation Sets

In [ ]:
test_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==2)
validation_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==1)

train = exp_block_df[~(test_mask | validation_mask)]
validation = exp_block_df[validation_mask]
test = exp_block_df[test_mask]
train.shape, validation.shape, test.shape

((11855088, 15), (595200, 15), (518400, 15))

In [ ]:
!pip install statsforecast utilsforecast > /dev/null
!pip install datasetsforecast > /dev/null

In [ ]:
import statsforecast
from functools import partial
from statsforecast.core import StatsForecast
from utilsforecast.plotting import plot_series
from utilsforecast.evaluation import evaluate
from statsforecast.models import (
    Naive, SeasonalNaive, HistoricAverage, WindowAverage, SeasonalWindowAverage,
    RandomWalkWithDrift, HoltWinters, #ETS,
    AutoETS, AutoARIMA, ARIMA, AutoTheta, DynamicTheta, DynamicOptimizedTheta,
    Theta, OptimizedTheta, TBATS, AutoTBATS, MSTL
)

In [ ]:
train_df = train[['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
validation_df = validation[
    ['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
test_df = test[['LCLid', 'timestamp', 'energy_consumption', 'frequency']]

In [ ]:
train_df.timestamp.min(), train_df.timestamp.max(), \
test_df.timestamp.min(), test_df.timestamp.max(), \
validation_df.timestamp.min(), validation_df.timestamp.max()

(Timestamp('2012-01-01 00:00:00'),
 Timestamp('2013-12-31 23:30:00'),
 Timestamp('2014-02-01 00:00:00'),
 Timestamp('2014-02-27 23:30:00'),
 Timestamp('2014-01-01 00:00:00'),
 Timestamp('2014-01-31 23:30:00'))

In [ ]:
freq_ = train_df.iloc[0]['frequency']
timeseries_train = train_df.loc[train_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]
timeseries_validation = validation_df.loc[validation_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]
timeseries_test = test_df.loc[test_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]

In [ ]:
predictions = pd.concat([timeseries_train, timeseries_validation])
predictions.head(), predictions.shape, predictions.dtypes

(       LCLid           timestamp  energy_consumption
 0  MAC000322 2012-03-07 00:00:00               0.125
 1  MAC000322 2012-03-07 00:30:00               0.104
 2  MAC000322 2012-03-07 01:00:00               0.133
 3  MAC000322 2012-03-07 01:30:00               0.145
 4  MAC000322 2012-03-07 02:00:00               0.109,
 (33408, 3),
 LCLid                         object
 timestamp             datetime64[ns]
 energy_consumption           float64
 dtype: object)

#### Baseline Forecasts - use built-in methods with the package

We review models that form strong baselines and try modern techniques in forecasting.

NIXTLA is one of the libraries we use to generate forecasts. NIXTLA has the flexibility to work directly with pandas.

NIXTLA looks for three columns: id-col column that uniquely identifies the time series, time-col the timestamp column, and target-col which is the column we want to forecast.

NIXTLA follows a scikit-learn style with .fit(), .predict() and adopts a .forecast() method, which is memory-efficient method that doesn't store the partial model outputs, whereas the scikit-learn interface stores the fitted models. For added flexibility, we use a list of metrics to get multiple measurements for each forecast. For MASE, the training set is also included.

For ease of experimentation, we encapsulated all of this into evaluate_performance method to return the predictions and the calculated metrics in a DataFrame.

In [ ]:
def evaluate_performance(ts_train, ts_test, models, metrics, freq, level, id_col, time_col, target_col, h, metric_df):
    if metric_df is None:
        metric_df = pd.DataFrame()

    results = ts_test.copy()
    timing = {}

    for model in models:
        model_name = model.__class__.__name__
        evaluation = {}

        start_time = time.time()

        sf = StatsForecast(
            models = [model], freq = freq, n_jobs = 1, fallback_model=Naive()
        )

        y_pred = sf.forecast(
            df = ts_train, h = h, level = level, id_col = id_col, time_col = time_col, target_col = target_col
        )

        duration = time.time() - start_time
        timing[model_name] = duration

        results = results.merge(y_pred, how='left', on=[id_col, time_col])

        ids = ts_train[id_col].unique()
        for id in ids:
            temp_results = results[results[id_col] == id]
            temp_train = ts_train[ts_train[id_col] == id]

            for metric in metrics:
                metric_name = metric.__name__
                if metric_name == 'mase':
                    evaluation[metric_name] = metric(temp_results[target_col], temp_results[model_name], temp_train[target_col])
                else:
                    evaluation[metric_name] = metric(temp_results[target_col], temp_results[model_name])

                evaluation[id_col] = id
                evaluation['Time Elapsed'] = timing[model_name]
                evaluation['Model'] = model_name

                temp_df = pd.DataFrame(evaluation, index=[0])
                temp_df['Model'] = model_name
                metric_df = pd.concat([metric_df, temp_df])

    return results, metric_df

In [ ]:
''' add code to utility for eval-performance '''
import time

#### Naive Forecast

Naïve forecast is just the last/most recent observation in a time series. If the latest observation in a time series is 10, then the forecast for all future timesteps is 10. This is implemented using the Naive class in NIXTLA.



In [ ]:
metrics = pd.DataFrame()

results, metrics = evaluate_performance(
    ts_train=timeseries_train,
    ts_test=timeseries_validation,
    models=[Naive()],
    metrics=[mase, mae, mse, forecast_bias],
    freq=freq_,
    level=[],
    id_col='LCLid',
    time_col='timestamp',
    target_col='energy_consumption',
    h=len(timeseries_validation),
    metric_df=metrics
)

In [ ]:
model_name = ['Naive']
model_display_name = ['Naive']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,0.960906,MAC000322,0.125081,Naive,NaN,NaN,NaN
0,0.960906,MAC000322,0.125081,Naive,0.060111,NaN,NaN
0,0.960906,MAC000322,0.125081,Naive,0.060111,0.010746,NaN
0,0.960906,MAC000322,0.125081,Naive,0.060111,0.010746,3.555109


#### Seasonal Naive Forecast

A seasonal naive forecast is a twist on the simple naive method. In the naive method, we take the last observation $(Yt-1)$, whereas in seasonal naïve, we take the $Yt-k$ observation. So, we look back $k$ steps for each forecast, this enables the algorithm to mimic the last seasonality cycle.

In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [SeasonalNaive(season_length=48*7)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['SeasonalNaive']
model_display_name = ['SeasonalNaive']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,1.2756,MAC000322,0.106611,SeasonalNaive,NaN,NaN,NaN
0,1.2756,MAC000322,0.106611,SeasonalNaive,0.079797,NaN,NaN
0,1.2756,MAC000322,0.106611,SeasonalNaive,0.079797,0.017751,NaN
0,1.2756,MAC000322,0.106611,SeasonalNaive,0.079797,0.017751,15.270441


#### Moving average forecast

While a naïve forecast memorizes the most recent past, it also memorizes the noise at any timestep. A moving average forecast is another method that tries to overcome the pure memorization of the naïve method. Instead of taking the latest observation, it takes the mean of the latest n steps as the forecast.

In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [WindowAverage(window_size=48)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['WindowAverage']
model_display_name = ['WindowAverage']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,1.011826,MAC000322,0.088453,WindowAverage,NaN,NaN,NaN
0,1.011826,MAC000322,0.088453,WindowAverage,0.063296,NaN,NaN
0,1.011826,MAC000322,0.088453,WindowAverage,0.063296,0.010798,NaN
0,1.011826,MAC000322,0.088453,WindowAverage,0.063296,0.010798,-7.817351


#### Exponential Smoothing Forecast

It is one of the most popular methods for generating forecasts. There are a few different variants of ETS—single exponential smoothing, double exponential smoothing, Holt-Winters’ seasonal smoothing.

In the naïve method, we use the latest observation. On the other hand, the moving average method considers the last n observations to be equally important and takes the mean of them. ETS combines both these intuitions and says that all the history is important, but the recent history is more important. Therefore, the forecast is generated using a weighted average where the weights decrease exponentially as we move farther into the history.
- $f_t = \alpha \star y_{t-1} + \alpha \star(1- \alpha)\star y_{t-2}$ + $\alpha \star (1 -\alpha)^2 \star y_{t-3}$.

Simple exponential smoothing (SES) is when we apply this smoothing procedure to the history.
This is more suited for time series that have no trends or seasonality, and the forecast is going to be a flat line. The forecast is given by
- $f_t = \alpha \star y_{t-1} + \alpha \star(1- \alpha)\star f_{t-1}$.

Double exponential smoothing (DES) extends the smoothing idea to model trends as well. It has two smoothing equations—one for the level and the other for the trend. Once we estimate level and trend, we combine them. This forecast is not necessarily flat because the estimated trend is used to extrapolate it into the future.

Triple exponential smoothing or Holt-Winters’ (HW) takes this one step forward by including another smoothing term to model seasonality.

In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [HoltWinters(error_type = 'A', season_length = 48)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['HoltWinters']
model_display_name = ['HoltWinters']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,0.960906,MAC000322,0.159328,HoltWinters,NaN,NaN,NaN
0,0.960906,MAC000322,0.159328,HoltWinters,0.060111,NaN,NaN
0,0.960906,MAC000322,0.159328,HoltWinters,0.060111,0.010746,NaN
0,0.960906,MAC000322,0.159328,HoltWinters,0.060111,0.010746,3.555109


#### Model AutoETS



In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [AutoETS(model = 'AAA',season_length = 48)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['AutoETS']
model_display_name = ['AutoETS']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,0.960906,MAC000322,0.100208,AutoETS,NaN,NaN,NaN
0,0.960906,MAC000322,0.100208,AutoETS,0.060111,NaN,NaN
0,0.960906,MAC000322,0.100208,AutoETS,0.060111,0.010746,NaN
0,0.960906,MAC000322,0.100208,AutoETS,0.060111,0.010746,3.555109


#### ARIMA

ARIMA models are the other class of methods like ETS that have stood the test of time and are one of the most popular classical methods of forecasting. The ETS family of methods is modeled around trend and seasonality, while ARIMA relies on autocorrelation (the correlation of $y_t$ with $y_{t-1}$, $y_{t-2}$, and so on).

The $AR (p)$ models, use linear regression with p previous timesteps or $p$ lags.
- $y_t = c + phi_1 \star y_{t-1} + phi_2 \star y_{t-2} + ... + $ $phi_p \star y_{t-p} + eps_t$, where, $c$ is the intercept and $eps_t$ is the noise or error at timestep t.

Next are MA (q) models, in which, instead of past observed values, we use the past $q$ errors in the forecast.
- $y_t = c + theta_1 \star eps_{t-1}$ + $theta_2 \star eps_{t-2} + ... $ +  $theta_q \star eps_{t-q}$, where, $c$ is the intercept and $eps$ is the white noise.

This is not typically used on its own but in conjunction with $AR (p)$ models, which makes the $ARMA (p, q)$ models. ARMA (AutoRegressive Moving Average) models are defined as $y_t = AR (p) + MA (q)$.

In all the ARIMA models, there is one underlying assumption, the time series is stationary. Take the difference of successive values for differencing. Sometimes, we difference once, while other times, we perform successive differencing before the time series becomes stationary. The number of times we perform differencing is the order of differencing. The I in ARIMA, for Integrated, defines the order of differencing before the series becomes stationary is denoted by d.

##### ARIMA ($p, d, q$)

The complete $ARIMA (p, d, q)$ model says that we do the dth order of differencing and then consider the last p terms in an autoregressive manner, and then include the last q moving average terms to come up with the forecast.

##### Seasonal ARIMA ($p, d, q$)

Taking the same concepts on a seasonal cycle, with p, q, q modified on the seasonal period, we get Seasonal ARIMA ($p, q, d$).

In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [ARIMA(order = (2,1,1), seasonal_order = (1,1,1), season_length = 48)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['ARIMA']
model_display_name = ['ARIMA']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,143.101743,MAC000322,116.837616,ARIMA,NaN,NaN,NaN
0,143.101743,MAC000322,116.837616,ARIMA,8.951943,NaN,NaN
0,143.101743,MAC000322,116.837616,ARIMA,8.951943,105.230962,NaN
0,143.101743,MAC000322,116.837616,ARIMA,8.951943,105.230962,8633.644878


With NIXTLA, both ETS and ARIMA capture both the seasonality and the peaks, the resulting MAE scores are also very similar. Next, we look at another method—the Theta forecast.

#### Theta



Theta forecast method relies on parameter theta, that amplifies or smooths the local curvature of a time series. The smoothed lines are called theta. This is a method of decomposition approach to forecasting.

The main steps that are involved in the Theta forecast implemented in NIXTLA
- Deseasonalization: Apply classical multiplicative decomposition to remove the seasonal component from the time series, focusing on the analysis on the underlying trend and cyclical components.
- Theta Coefficients Application: Decompose and deseasonalize into two Theta lines using coefficients $theta_{1}$ and $theta_{2}$. These coefficients modify the second difference of the time series to discard or accentuate local fluctuations.
- Extrapolation of Theta Lines: Treat each Theta line as a separate time series and forecast them. This using linear regression for the Theta line where $theta_1 = 0$ giving a straight line, and simple exponential smoothing for the Theta line where $theta_2 = 0$.
- Recomposition: Combine the forecasts from two Theta lines. The original method uses equal weighting for both lines, which integrates long-term tends and short-term movemets effectively.
- Reseasonalize if the data was deseasonalized.

In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [Theta(season_length =48, decomposition_type = 'additive' )],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['Theta']
model_display_name = ['Theta']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
rmse, smape

#### TBATS

Sometimes, a time series has more than one seasonality pattern or a non-integer seasonal period (complex seasonality). Additionally, most time series models are designed for smaller integer seasonal periods, such as monthly or quarterly data, but yearly seasonality can pose a problem. TBATS was meant to combat these challenges that pose problems for many forecasting models. However, with any automated approach, at times is susceptible to poor forecasts.

TBATS stands for trigonometric seasonality, box-cox transformation, ARMA errors, trend and seasonal components. TBATS is from the state space model family, where forecasting models, the observed time series is assumed to be a combination of the underlying state variables and a measurement equation that relates the state variables to the observed data. The state variables capture the underlying patterns, trends, and relationships in the data.

The order of operations in using TBATS
- Box-Cox transformation
- Exponentially smoothed trend
- Seasonal decomposition using Fourier series (trigonometric seasonality)
- AutoRegressive Moving Average (ARMA)
- Parameter estimation through a likelihood-based approach

In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [TBATS(season_length = 48, use_boxcox=False)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['TBATS']
model_display_name = ['TBATS']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,1.93201,MAC000322,1.170573,TBATS,NaN,NaN,NaN
0,1.93201,MAC000322,1.170573,TBATS,0.12086,NaN,NaN
0,1.93201,MAC000322,1.170573,TBATS,0.12086,0.020722,NaN
0,1.93201,MAC000322,1.170573,TBATS,0.12086,0.020722,-90.322055


##### Parameter optimization
To select the optimal parameter space, TBATS will fit several models and automatically select the best parameters.

#### MSTL

Trend and cyclical components can be extracted using LOESS regression. If we fit a simple model on the trend values, we can use it to extrapolate to the future. And the seasonality component can easily be extrapolated because it is supposed to be a repeating pattern. Combining these, we get a forecasting model that works well.

The MSTL method in NIXTLA applies the LOESS technique to decompose a time series into its various seasonal components. Following this decomposition, it employs a specialized non-seasonal model to forecast the trend, and a Seasonal Naive model to predict each of the seasonal components.



In [ ]:
metrics = pd.DataFrame()

results, metrics = (
    evaluate_performance(
        ts_train=timeseries_train,
        ts_test=timeseries_validation,
        models = [MSTL(season_length = 48)],
        metrics = [mase, mae, mse, forecast_bias],
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = len(timeseries_validation),
        metric_df=metrics  )
)

In [ ]:
model_name = ['MSTL']
model_display_name = ['MSTL']

fig = plot_forecast(
    results, forecast_columns=model_name,forecast_display_names=model_display_name)

fig = format_plot(fig)
fig.update_layout(title_text=f"{model_name[0]}: "\
                  f"MAE: {metrics.loc[metrics.Model==model_name[0]][['mae']].iloc[0].item():.4f} | "\
                  f"MASE: {metrics.loc[metrics.Model==model_name[0]][['mase']].iloc[0].item():.4f} | "\
                  f"BIAS: {metrics.loc[metrics.Model==model_name[0]][['forecast_bias']].iloc[0].item():.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])

fig.show()

In [ ]:
metrics

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,0.960906,MAC000322,0.089476,MSTL,NaN,NaN,NaN
0,0.960906,MAC000322,0.089476,MSTL,0.060111,NaN,NaN
0,0.960906,MAC000322,0.089476,MSTL,0.060111,0.010746,NaN
0,0.960906,MAC000322,0.089476,MSTL,0.060111,0.010746,3.555109


#### Forecasting for Validation period for several models

In [ ]:
validation_models = [
    #ARIMA(order = (2,1,1), seasonal_order = (1,1,1), season_length = 48),
    AutoETS(model = 'AAA',season_length = 48),
    TBATS(season_length = 48, use_boxcox=False)
]
validation_models_names = [
    model.__class__.__name__ for model in validation_models]
metric_df = pd.DataFrame([])
h_val = 1488

aggregated_val_metrics = pd.DataFrame()
baseline_val_pred_df, aggregated_val_metrics = (
    evaluate_performance(
        timeseries_train[["LCLid","timestamp","energy_consumption"]],
        timeseries_validation[["LCLid","timestamp","energy_consumption"]],
        models =validation_models,
        metrics = [mase, mae, mse, forecast_bias], # rmse, smape
        freq = freq_,
        level = [] ,
        id_col = 'LCLid',
        time_col = 'timestamp',
        target_col = 'energy_consumption',
        h = h_val,
        metric_df = aggregated_val_metrics
    )
)

In [ ]:
aggregated_val_metrics.head()

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,0.960906,MAC000322,0.124938,AutoETS,NaN,NaN,NaN
0,0.960906,MAC000322,0.124938,AutoETS,0.060111,NaN,NaN
0,0.960906,MAC000322,0.124938,AutoETS,0.060111,0.010746,NaN
0,0.960906,MAC000322,0.124938,AutoETS,0.060111,0.010746,3.555109
0,1.932010,MAC000322,1.159333,TBATS,NaN,NaN,NaN


In [ ]:
baseline_val_pred_df[baseline_val_pred_df.LCLid =='MAC000322'].head()

,LCLid,timestamp,energy_consumption,AutoETS,TBATS
0,MAC000322,2014-01-01 00:00:00,0.056,0.1,0.160797
1,MAC000322,2014-01-01 00:30:00,0.055,0.1,0.162791
2,MAC000322,2014-01-01 01:00:00,0.104,0.1,0.164392
3,MAC000322,2014-01-01 01:30:00,0.039,0.1,0.165264
4,MAC000322,2014-01-01 02:00:00,0.011,0.1,0.164904


In [ ]:
aggregated_val_metrics[aggregated_val_metrics.Model =='TBATS'].sort_values(by='mase', ascending=True)

,mase,LCLid,Time Elapsed,Model,mae,mse,forecast_bias
0,1.93201,MAC000322,1.159333,TBATS,NaN,NaN,NaN
0,1.93201,MAC000322,1.159333,TBATS,0.12086,NaN,NaN
0,1.93201,MAC000322,1.159333,TBATS,0.12086,0.020722,NaN
0,1.93201,MAC000322,1.159333,TBATS,0.12086,0.020722,-90.322055


Decomposing Time Series

In [ ]:
exp_block_df.head(2)

,timestamp,LCLid,energy_consumption,frequency,series_length,stdorToU,Acorn,Acorn_grouped,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,2012-03-07 00:00:00,MAC000322,0.125,30min,34704,Std,ACORN-D,Affluent,1024.14,2.65,3.76,rain,partly-cloudy-night,0.8,Partly Cloudy
1,2012-03-07 00:30:00,MAC000322,0.104,30min,34704,Std,ACORN-D,Affluent,1024.14,2.65,3.76,rain,partly-cloudy-night,0.8,Partly Cloudy
